# 03. 현대 Transformer block과 생성 비용

목표: RMSNorm, SwiGLU, residual update와 autoregressive generation을 조합하고 KV-cache memory를 계산합니다. Random weight toy model이므로 생성 품질은 평가 대상이 아닙니다.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
np.set_printoptions(precision=3, suppress=True)

## 1. RMSNorm과 SwiGLU

RMSNorm은 vector의 root-mean-square로 scale을 맞춥니다. SwiGLU는 한 projection을 SiLU gate로 사용해 다른 projection을 조절합니다.

In [ ]:
def rms_norm(x, weight, eps=1e-6):
    rms = np.sqrt(np.mean(x * x, axis=-1, keepdims=True) + eps)
    return (x / rms) * weight

def silu(x):
    return x / (1.0 + np.exp(-np.clip(x, -30, 30)))

def swiglu(x, w_gate, w_value, w_down):
    return (silu(x @ w_gate) * (x @ w_value)) @ w_down

d_model, d_ff = 8, 20
norm_weight = np.ones(d_model)
w_gate = rng.normal(scale=0.15, size=(d_model, d_ff))
w_value = rng.normal(scale=0.15, size=(d_model, d_ff))
w_down = rng.normal(scale=0.15, size=(d_ff, d_model))
sample = rng.normal(size=(3, d_model))
normalized = rms_norm(sample, norm_weight)
ffn_output = swiglu(normalized, w_gate, w_value, w_down)
print('RMS after normalization:', np.sqrt(np.mean(normalized ** 2, axis=-1)))
assert normalized.shape == sample.shape
assert ffn_output.shape == sample.shape
assert np.allclose(np.sqrt(np.mean(normalized ** 2, axis=-1)), 1.0, atol=1e-5)

## 2. Pre-norm residual block

Attention과 FFN은 normalized input을 읽고 결과를 residual stream에 더합니다.

In [ ]:
def softmax_rows(values):
    shifted = values - values.max(axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=-1, keepdims=True)

wq, wk, wv, wo = [rng.normal(scale=0.15, size=(d_model, d_model)) for _ in range(4)]

def causal_self_attention(x):
    q, k, v = x @ wq, x @ wk, x @ wv
    scores = q @ k.T / np.sqrt(d_model)
    mask = np.triu(np.ones_like(scores, dtype=bool), k=1)
    weights = softmax_rows(np.where(mask, -1e30, scores))
    return (weights @ v) @ wo

def transformer_block(x):
    after_attention = x + causal_self_attention(rms_norm(x, norm_weight))
    return after_attention + swiglu(rms_norm(after_attention, norm_weight), w_gate, w_value, w_down)

block_output = transformer_block(sample)
print('input/output shapes:', sample.shape, block_output.shape)
assert block_output.shape == sample.shape
assert not np.allclose(block_output, sample)

## 3. Toy autoregressive loop

마지막 위치를 vocabulary logits로 투영하고 token 하나를 뽑아 다시 입력에 붙입니다. 실제 구현은 KV cache로 과거 K/V를 재사용하지만 여기서는 이해를 위해 전체 prefix를 다시 계산합니다.

In [ ]:
vocab = ['<eos>', 'token', 'attention', 'vector', 'model', 'works']
embedding = rng.normal(scale=0.2, size=(len(vocab), d_model))
unembedding = embedding.T  # weight tying의 toy 형태

def sample_token(logits, temperature=0.8):
    probabilities = softmax_rows((logits / temperature)[None, :])[0]
    return int(rng.choice(len(logits), p=probabilities)), probabilities

def generate(prompt_ids, max_new_tokens=5):
    ids = list(prompt_ids)
    trace = []
    for _ in range(max_new_tokens):
        hidden = transformer_block(embedding[ids])
        logits = hidden[-1] @ unembedding
        next_id, probabilities = sample_token(logits)
        trace.append((next_id, float(probabilities[next_id])))
        ids.append(next_id)
        if next_id == 0:
            break
    return ids, trace

generated, trace = generate([1, 4])
print('tokens:', [vocab[i] for i in generated])
print('selected probabilities:', trace)
assert generated[:2] == [1, 4]
assert len(generated) <= 7

## 4. KV-cache memory audit

Decoder KV cache의 근사 byte 수는 `layers × tokens × 2(K,V) × kv_heads × head_dim × bytes_per_value × batch`입니다. GQA의 효과를 head 수만 바꿔 비교합니다.

In [ ]:
def kv_cache_bytes(layers, tokens, kv_heads, head_dim, bytes_per_value=2, batch=1):
    return layers * tokens * 2 * kv_heads * head_dim * bytes_per_value * batch

config = {'layers': 80, 'tokens': 8192, 'head_dim': 128, 'bytes_per_value': 2}
full_mha = kv_cache_bytes(kv_heads=64, **config)
gqa = kv_cache_bytes(kv_heads=8, **config)
print(f'full MHA: {full_mha / 2**30:.2f} GiB')
print(f'GQA:      {gqa / 2**30:.2f} GiB')
print(f'reduction: {full_mha / gqa:.1f}x')
assert full_mha / gqa == 8
assert kv_cache_bytes(kv_heads=8, **{**config, 'tokens': 16384}) == 2 * gqa

## 정리

현대 decoder block의 핵심 부품을 결합했습니다. 품질뿐 아니라 context 길이, KV head 수, precision과 batch가 serving memory를 결정합니다. 실제 제품에서는 prefill·decode latency, kernel, quantization과 distributed execution도 함께 측정해야 합니다.